# Cost Optimization for AI Systems

This notebook covers practical cost optimization techniques:
- Model routing: using cheap models for simple tasks
- Token counting and cost tracking
- Streaming for perceived performance
- Combining caching + routing for cost savings
- Building a cost-aware AI pipeline

## 1. Setup

In [ ]:
import time
import asyncio
import hashlib
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict

print("Dependencies loaded.")

## 2. Token Counting and Cost Tracking

Track token usage and costs across different models.

In [ ]:
@dataclass
class ModelPricing:
    name: str
    input_price_per_million: float  # USD per 1M tokens
    output_price_per_million: float
    max_tokens: int
    speed_rating: str  # "fast", "medium", "slow"


# Model pricing (approximate, check current rates)
MODELS = {
    "gpt-3.5-turbo": ModelPricing(
        name="gpt-3.5-turbo",
        input_price_per_million=0.50,
        output_price_per_million=1.50,
        max_tokens=4096,
        speed_rating="fast"
    ),
    "gpt-4o-mini": ModelPricing(
        name="gpt-4o-mini",
        input_price_per_million=0.15,
        output_price_per_million=0.60,
        max_tokens=16384,
        speed_rating="fast"
    ),
    "gpt-4o": ModelPricing(
        name="gpt-4o",
        input_price_per_million=2.50,
        output_price_per_million=10.00,
        max_tokens=16384,
        speed_rating="medium"
    ),
    "claude-3-haiku": ModelPricing(
        name="claude-3-haiku",
        input_price_per_million=0.25,
        output_price_per_million=1.25,
        max_tokens=4096,
        speed_rating="fast"
    ),
    "claude-3-5-sonnet": ModelPricing(
        name="claude-3-5-sonnet",
        input_price_per_million=3.00,
        output_price_per_million=15.00,
        max_tokens=8192,
        speed_rating="medium"
    ),
}


class CostTracker:
    """Track token usage and costs across models."""

    def __init__(self):
        self.usage: dict[str, dict] = defaultdict(lambda: {
            "input_tokens": 0,
            "output_tokens": 0,
            "calls": 0,
            "total_cost": 0.0
        })

    def estimate_tokens(self, text: str) -> int:
        """Rough token estimate (4 chars per token for English)."""
        return len(text) // 4

    def record_call(
        self,
        model: str,
        input_text: str,
        output_text: str
    ) -> float:
        """Record an API call and return its cost."""
        pricing = MODELS.get(model)
        if not pricing:
            return 0.0

        input_tokens = self.estimate_tokens(input_text)
        output_tokens = self.estimate_tokens(output_text)

        cost = (
            input_tokens * pricing.input_price_per_million / 1_000_000 +
            output_tokens * pricing.output_price_per_million / 1_000_000
        )

        self.usage[model]["input_tokens"] += input_tokens
        self.usage[model]["output_tokens"] += output_tokens
        self.usage[model]["calls"] += 1
        self.usage[model]["total_cost"] += cost

        return cost

    def summary(self) -> dict:
        """Generate cost summary."""
        total_cost = 0.0
        total_calls = 0

        model_details = {}
        for model, data in self.usage.items():
            model_details[model] = {
                "calls": data["calls"],
                "input_tokens": data["input_tokens"],
                "output_tokens": data["output_tokens"],
                "cost": f"${data['total_cost']:.4f}",
            }
            total_cost += data["total_cost"]
            total_calls += data["calls"]

        return {
            "total_cost": f"${total_cost:.4f}",
            "total_calls": total_calls,
            "by_model": model_details,
        }


# Demo
tracker = CostTracker()

# Simulate some API calls
calls = [
    ("gpt-3.5-turbo", "Hello, how are you?", "I'm doing well, thanks!"),
    ("gpt-4o", "Explain quantum computing in detail.", "Quantum computing uses quantum bits..." * 50),
    ("gpt-3.5-turbo", "What is 2+2?", "4"),
    ("gpt-4o-mini", "Summarize this text.", "This is a summary."),
    ("gpt-4o", "Write a complex algorithm.", "def complex_algorithm()..." * 30),
]

for model, input_text, output_text in calls:
    cost = tracker.record_call(model, input_text, output_text)
    print(f"{model}: ${cost:.6f}")

print("\nCost Summary:")
summary = tracker.summary()
print(f"Total cost: {summary['total_cost']}")
print(f"Total calls: {summary['total_calls']}")
for model, details in summary['by_model'].items():
    print(f"  {model}: {details['calls']} calls, {details['cost']}")

## 3. Model Routing

Route queries to appropriate models based on complexity.

In [ ]:
class QueryComplexityClassifier:
    """Classify query complexity for model routing."""

    SIMPLE_PATTERNS = [
        "what is", "who is", "when did", "where is",
        "define", "meaning of", "how many", "how much",
        "yes or no", "true or false",
    ]

    MEDIUM_PATTERNS = [
        "explain", "describe", "compare", "summarize",
        "list", "give me", "tell me about",
    ]

    COMPLEX_PATTERNS = [
        "analyze", "evaluate", "critique", "design",
        "write code", "implement", "create a plan",
        "step by step", "in detail", "comprehensive",
    ]

    def classify(self, query: str) -> str:
        """Classify query complexity."""
        query_lower = query.lower()

        # Check for complex patterns first
        for pattern in self.COMPLEX_PATTERNS:
            if pattern in query_lower:
                return "complex"

        # Check for medium patterns
        for pattern in self.MEDIUM_PATTERNS:
            if pattern in query_lower:
                return "medium"

        # Check for simple patterns
        for pattern in self.SIMPLE_PATTERNS:
            if pattern in query_lower:
                return "simple"

        # Default based on length
        words = len(query.split())
        if words < 10:
            return "simple"
        elif words < 30:
            return "medium"
        else:
            return "complex"


class ModelRouter:
    """Route queries to appropriate models based on complexity."""

    ROUTING_TABLE = {
        "simple": "gpt-4o-mini",
        "medium": "gpt-4o-mini",
        "complex": "gpt-4o",
    }

    def __init__(self, classifier: QueryComplexityClassifier = None):
        self.classifier = classifier or QueryComplexityClassifier()
        self.stats = defaultdict(int)

    def route(self, query: str) -> str:
        """Route query to appropriate model."""
        complexity = self.classifier.classify(query)
        model = self.ROUTING_TABLE[complexity]
        self.stats[complexity] += 1
        return model

    def get_stats(self) -> dict:
        return dict(self.stats)


# Demo
router = ModelRouter()

queries = [
    "What is Python?",
    "Who is Elon Musk?",
    "Explain the difference between TCP and UDP.",
    "Summarize the key points of this article.",
    "Design a scalable microservices architecture for a chat application.",
    "Implement a red-black tree in Python with detailed comments.",
    "Compare React, Vue, and Angular for a large enterprise application.",
    "How many planets are in the solar system?",
]

print("Query Routing:")
print("="*80)
for query in queries:
    model = router.route(query)
    complexity = router.classifier.classify(query)
    print(f"[{complexity:>8}] -> {model:<15} | {query[:50]}")

print(f"\nRouting stats: {router.get_stats()}")

## 4. Cost Simulation: Routing vs No Routing

Compare costs with and without model routing.

In [ ]:
async def simulate_llm_call(model: str, prompt: str) -> str:
    """Simulate an LLM call with model-specific latency."""
    pricing = MODELS[model]
    latency = {"fast": 0.1, "medium": 0.3, "slow": 0.8}[pricing.speed_rating]
    await asyncio.sleep(latency)
    # Simulate output length based on model capability
    output_length = {"gpt-3.5-turbo": 50, "gpt-4o-mini": 60, "gpt-4o": 100}[model]
    return "x" * output_length


async def cost_comparison(queries: list[str]):
    """Compare costs: always using expensive model vs routing."""

    # Scenario 1: Always use expensive model
    expensive_tracker = CostTracker()
    start = time.time()
    for query in queries:
        response = await simulate_llm_call("gpt-4o", query)
        expensive_tracker.record_call("gpt-4o", query, response)
    expensive_time = time.time() - start

    # Scenario 2: Use model routing
    routing_tracker = CostTracker()
    router = ModelRouter()
    start = time.time()
    for query in queries:
        model = router.route(query)
        response = await simulate_llm_call(model, query)
        routing_tracker.record_call(model, query, response)
    routing_time = time.time() - start

    # Compare
    print("Cost Comparison: Always GPT-4o vs Model Routing")
    print("="*60)

    expensive_summary = expensive_tracker.summary()
    routing_summary = routing_tracker.summary()

    expensive_cost = float(expensive_summary["total_cost"].replace("$", ""))
    routing_cost = float(routing_summary["total_cost"].replace("$", ""))
    savings = expensive_cost - routing_cost
    savings_pct = (savings / expensive_cost * 100) if expensive_cost > 0 else 0

    print(f"\nAlways GPT-4o:")
    print(f"  Cost: {expensive_summary['total_cost']}")
    print(f"  Time: {expensive_time:.2f}s")

    print(f"\nWith Routing:")
    print(f"  Cost: {routing_summary['total_cost']}")
    print(f"  Time: {routing_time:.2f}s")
    for model, details in routing_summary['by_model'].items():
        print(f"    {model}: {details['calls']} calls")

    print(f"\nSavings: ${savings:.4f} ({savings_pct:.1f}%)")

    return {
        "expensive_cost": expensive_cost,
        "routing_cost": routing_cost,
        "savings": savings,
        "savings_pct": savings_pct,
    }


test_queries = [
    "What is Python?",
    "Who is the CEO of Apple?",
    "Explain how neural networks learn.",
    "Summarize this research paper.",
    "Design a fault-tolerant distributed system.",
    "When was the internet invented?",
    "Compare PostgreSQL and MongoDB.",
    "Write a parser for JSON.",
    "How do I reset my password?",
    "Evaluate the pros and cons of microservices.",
]

results = await cost_comparison(test_queries)

## 5. Caching + Routing Combined

The biggest cost savings come from combining caching with routing.

In [ ]:
class CostOptimizedPipeline:
    """Pipeline combining caching + routing for maximum cost savings."""

    def __init__(self):
        self.cache: dict[str, str] = {}  # simple in-memory cache
        self.router = ModelRouter()
        self.tracker = CostTracker()
        self.cache_hits = 0
        self.cache_misses = 0

    def _cache_key(self, query: str) -> str:
        return hashlib.sha256(query.lower().strip().encode()).hexdigest()[:16]

    async def process(self, query: str) -> dict:
        """Process query with caching and routing."""
        # Step 1: Check cache
        key = self._cache_key(query)
        if key in self.cache:
            self.cache_hits += 1
            return {
                "response": self.cache[key],
                "model": "cached",
                "cost": 0.0,
                "cached": True,
            }

        self.cache_misses += 1

        # Step 2: Route to appropriate model
        model = self.router.route(query)

        # Step 3: Generate response
        response = await simulate_llm_call(model, query)

        # Step 4: Track cost
        cost = self.tracker.record_call(model, query, response)

        # Step 5: Cache the response
        self.cache[key] = response

        return {
            "response": response,
            "model": model,
            "cost": cost,
            "cached": False,
        }

    def stats(self) -> dict:
        total = self.cache_hits + self.cache_misses
        return {
            "cache_hit_rate": f"{self.cache_hits / total:.1%}" if total > 0 else "N/A",
            "cache_hits": self.cache_hits,
            "cache_misses": self.cache_misses,
            "cost_summary": self.tracker.summary(),
        }


# Demo
pipeline = CostOptimizedPipeline()

# Simulate workload with repeated queries
workload = [
    "What is Python?",
    "Explain machine learning.",
    "What is Python?",  # repeat
    "Who is Elon Musk?",
    "Explain machine learning.",  # repeat
    "Design a scalable system.",
    "What is Python?",  # repeat
    "Summarize this document.",
    "Who is Elon Musk?",  # repeat
    "What is Python?",  # repeat
]

print("Processing workload with cost-optimized pipeline:\n")
for i, query in enumerate(workload, 1):
    result = await pipeline.process(query)
    cached_label = "[CACHED]" if result["cached"] else "[NEW]"
    print(f"{i:2}. {cached_label} {result['model']:<15} ${result['cost']:.6f} | {query[:40]}")

print("\nPipeline Statistics:")
stats = pipeline.stats()
print(f"  Cache hit rate: {stats['cache_hit_rate']}")
print(f"  Total cost: {stats['cost_summary']['total_cost']}")
for model, details in stats['cost_summary']['by_model'].items():
    print(f"    {model}: {details['calls']} calls, {details['cost']}")

## 6. Streaming for Perceived Performance

Streaming improves user experience without reducing actual cost.

In [ ]:
async def simulate_streaming_response(query: str, model: str):
    """Simulate streaming response with time-to-first-token (TTFT)."""
    pricing = MODELS[model]
    ttft = {"fast": 0.15, "medium": 0.4, "slow": 0.8}[pricing.speed_rating]

    # Simulate TTFT
    await asyncio.sleep(ttft)

    # Generate tokens
    tokens = [f"token_{i}" for i in range(20)]
    for token in tokens:
        yield token
        await asyncio.sleep(0.02)  # inter-token latency


async def compare_streaming_vs_buffered(query: str, model: str):
    """Compare perceived latency of streaming vs buffered."""

    # Buffered: wait for full response
    start = time.time()
    full_response = ""
    async for token in simulate_streaming_response(query, model):
        full_response += token + " "
    buffered_time = time.time() - start

    # Streaming: measure TTFT
    start = time.time()
    first_token_time = None
    token_count = 0
    async for token in simulate_streaming_response(query, model):
        if first_token_time is None:
            first_token_time = time.time() - start
        token_count += 1
    streaming_total = time.time() - start

    print(f"Model: {model}")
    print(f"  Buffered total time: {buffered_time*1000:.0f}ms")
    print(f"  Streaming TTFT: {first_token_time*1000:.0f}ms")
    print(f"  Streaming total: {streaming_total*1000:.0f}ms")
    print(f"  Perceived improvement: {(buffered_time - first_token_time)/buffered_time*100:.0f}%")


print("Streaming vs Buffered Comparison:\n")
for model in ["gpt-3.5-turbo", "gpt-4o-mini", "gpt-4o"]:
    await compare_streaming_vs_buffered("Explain quantum computing", model)
    print()

## 7. Complete Cost-Optimized Pipeline

Combine all techniques into a production-ready pipeline.

In [ ]:
class ProductionAIPipeline:
    """Production pipeline with caching, routing, and cost tracking."""

    def __init__(self):
        self.cache: dict[str, dict] = {}  # key -> {response, model, timestamp}
        self.router = ModelRouter()
        self.tracker = CostTracker()
        self.cache_ttl = 3600  # 1 hour
        self.total_requests = 0
        self.total_saved = 0.0

    def _cache_key(self, query: str) -> str:
        return hashlib.sha256(query.lower().strip().encode()).hexdigest()[:16]

    def _is_cache_valid(self, entry: dict) -> bool:
        return (time.time() - entry["timestamp"]) < self.cache_ttl

    async def process(self, query: str) -> dict:
        """Process with full optimization stack."""
        self.total_requests += 1

        # 1. Cache check
        key = self._cache_key(query)
        if key in self.cache and self._is_cache_valid(self.cache[key]):
            entry = self.cache[key]
            return {
                "response": entry["response"],
                "model": "cached",
                "cost": 0.0,
                "latency_ms": 0,
                "source": "cache",
            }

        # 2. Route to model
        model = self.router.route(query)
        pricing = MODELS[model]

        # 3. Generate
        start = time.time()
        response = await simulate_llm_call(model, query)
        latency = (time.time() - start) * 1000

        # 4. Track cost
        cost = self.tracker.record_call(model, query, response)

        # 5. Cache
        self.cache[key] = {
            "response": response,
            "model": model,
            "timestamp": time.time(),
        }

        return {
            "response": response,
            "model": model,
            "cost": cost,
            "latency_ms": latency,
            "source": "generated",
        }

    def report(self) -> dict:
        """Generate optimization report."""
        cost_summary = self.tracker.summary()
        total_cost = float(cost_summary["total_cost"].replace("$", ""))

        # Estimate cost if all went to expensive model
        naive_cost = total_cost * 5  # rough estimate

        return {
            "total_requests": self.total_requests,
            "actual_cost": cost_summary["total_cost"],
            "estimated_naive_cost": f"${naive_cost:.4f}",
            "model_breakdown": cost_summary["by_model"],
            "optimizations": {
                "caching": "Avoids redundant API calls",
                "routing": "Uses cheaper models for simple queries",
            },
        }


# Demo
pipeline = ProductionAIPipeline()

# Simulate realistic workload
workload = [
    "What is Python?",
    "Explain machine learning in detail.",
    "What is Python?",
    "Who is the CEO of Tesla?",
    "Design a distributed caching system.",
    "Explain machine learning in detail.",
    "What is 2+2?",
    "Summarize this research paper on transformers.",
    "Who is the CEO of Tesla?",
    "What is Python?",
    "Implement a binary search tree.",
    "What is 2+2?",
]

print("Processing workload:\n")
for i, query in enumerate(workload, 1):
    result = await pipeline.process(query)
    source_tag = "[CACHE]" if result["source"] == "cache" else f"[{result['model']}"
    print(f"{i:2}. {source_tag:<20} ${result['cost']:.6f} | {query[:40]}")

print("\n" + "="*60)
print("OPTIMIZATION REPORT")
print("="*60)
report = pipeline.report()
print(f"Total requests: {report['total_requests']}")
print(f"Actual cost: {report['actual_cost']}")
print(f"Estimated naive cost: {report['estimated_naive_cost']}")
print(f"\nModel breakdown:")
for model, details in report['model_breakdown'].items():
    print(f"  {model}: {details['calls']} calls, {details['cost']}")

## Key Takeaways

1. **Model routing** reduces cost by 50-80% for mixed workloads
2. **Caching** eliminates cost entirely for repeated queries
3. **Combined caching + routing** provides maximum savings
4. **Streaming** improves perceived performance without reducing cost
5. **Track everything** — you can't optimize what you don't measure